# 00 – Build EC3D Paper-Aligned Dataset (No Unknown)

**Obiettivo**: Creare una versione "clean" del dataset EC3D che esclude la classe "Unknown" (SQUAT, instruction_id=10), allineata con la tabella del paper EC3D.

## Trasformazioni:
1. Filtro sequenze con exercise='SQUAT' e instruction_id=10
2. Remapping labels da 12 a 11 classi (shift delle classi 6-11 a 5-10)
3. Aggiornamento split cross-subject
4. Verifica allineamento con paper (362 sequenze totali)

## Output:
- `data/EC3D/ec3d_sequences_no_unknown.pkl`
- `data/EC3D/split_cross_subject_no_unknown.json`
- `data/EC3D/ec3d_no_unknown_info.json`


In [1]:
import sys
import pickle
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

# Root e path
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data" / "EC3D"

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"DATA_DIR: {DATA_DIR}")

# Aggiungi root al path per importare utils
sys.path.insert(0, str(ROOT_DIR))


ROOT_DIR: /home/giov/Scrivania/Tesi/pose-text-feedback-thesis
DATA_DIR: /home/giov/Scrivania/Tesi/pose-text-feedback-thesis/data/EC3D


## 1. Label Mapping Configuration

Definiamo la mappatura delle label:
- **Con Unknown (12 classi)**: 0-4 (Squat), 5 (Squat-Unknown), 6-8 (Lunges), 9-11 (Plank)
- **Senza Unknown (11 classi)**: 0-4 (Squat), 5-7 (Lunges), 8-10 (Plank)


In [2]:
# === LABEL MAPPING ===

# Mapping con Unknown (12 classi) - ORIGINALE
ID_TO_NAME_OLD = {
    0: "SQUAT - Correct",
    1: "SQUAT - Feet too wide",
    2: "SQUAT - Knees inward",
    3: "SQUAT - Not low enough",
    4: "SQUAT - Front bended",
    5: "SQUAT - Unknown",        # QUESTO VIENE RIMOSSO
    6: "LUNGES - Correct",
    7: "LUNGES - Not low enough",
    8: "LUNGES - Knees pass toes",
    9: "PLANK - Correct",
    10: "PLANK - Banana back",
    11: "PLANK - Rolled back",
}

# Mapping senza Unknown (11 classi) - PAPER-ALIGNED
ID_TO_NAME_NEW = {
    0: "SQUAT - Correct",
    1: "SQUAT - Feet too wide",
    2: "SQUAT - Knees inward",
    3: "SQUAT - Not low enough",
    4: "SQUAT - Front bended",
    # 5 = old 6: LUNGES - Correct
    5: "LUNGES - Correct",
    6: "LUNGES - Not low enough",
    7: "LUNGES - Knees pass toes",
    8: "PLANK - Correct",
    9: "PLANK - Banana back",
    10: "PLANK - Rolled back",
}

# Mapping: old_label -> new_label (label 5 viene RIMOSSA, 6-11 shiftano a 5-10)
OLD_TO_NEW_LABEL_MAP = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    # 5: UNKNOWN - sarà filtrato, non mappato
    6: 5,
    7: 6,
    8: 7,
    9: 8,
    10: 9,
    11: 10,
}

# Verifica mapping
print("📋 Mapping OLD -> NEW:")
for old_id, new_id in OLD_TO_NEW_LABEL_MAP.items():
    print(f"   {old_id:2d} ({ID_TO_NAME_OLD[old_id]:30s}) -> {new_id:2d} ({ID_TO_NAME_NEW[new_id]})")

print(f"\n⚠️  Label 5 ({ID_TO_NAME_OLD[5]}) verrà RIMOSSA")
print(f"\n📊 Classi: 12 -> 11")


📋 Mapping OLD -> NEW:
    0 (SQUAT - Correct               ) ->  0 (SQUAT - Correct)
    1 (SQUAT - Feet too wide         ) ->  1 (SQUAT - Feet too wide)
    2 (SQUAT - Knees inward          ) ->  2 (SQUAT - Knees inward)
    3 (SQUAT - Not low enough        ) ->  3 (SQUAT - Not low enough)
    4 (SQUAT - Front bended          ) ->  4 (SQUAT - Front bended)
    6 (LUNGES - Correct              ) ->  5 (LUNGES - Correct)
    7 (LUNGES - Not low enough       ) ->  6 (LUNGES - Not low enough)
    8 (LUNGES - Knees pass toes      ) ->  7 (LUNGES - Knees pass toes)
    9 (PLANK - Correct               ) ->  8 (PLANK - Correct)
   10 (PLANK - Banana back           ) ->  9 (PLANK - Banana back)
   11 (PLANK - Rolled back           ) -> 10 (PLANK - Rolled back)

⚠️  Label 5 (SQUAT - Unknown) verrà RIMOSSA

📊 Classi: 12 -> 11


## 2. Load Original Dataset


In [3]:
# Load original dataset
with open(DATA_DIR / "ec3d_sequences.pkl", "rb") as f:
    ec3d_orig = pickle.load(f)

# Load original split
with open(DATA_DIR / "split_cross_subject.json", "r") as f:
    split_orig = json.load(f)

sequences_orig = ec3d_orig["sequences"]  # list of np.array (T, 3, 25)
labels_orig = np.array(ec3d_orig["labels"])
meta_orig = ec3d_orig["meta"]

train_indices_orig = np.array(split_orig["train_indices"])
test_indices_orig = np.array(split_orig["test_indices"])

print(f"📥 Dataset ORIGINALE caricato:")
print(f"   Totale sequenze: {len(sequences_orig)}")
print(f"   Train: {len(train_indices_orig)}, Test: {len(test_indices_orig)}")
print(f"   Classi uniche: {sorted(np.unique(labels_orig).tolist())}")

# Distribuzione classi
print(f"\n📊 Distribuzione classi (originale):")
for cid in range(12):
    count = np.sum(labels_orig == cid)
    in_train = np.sum(labels_orig[train_indices_orig] == cid)
    in_test = np.sum(labels_orig[test_indices_orig] == cid)
    print(f"   [{cid:2d}] {ID_TO_NAME_OLD[cid]:30s}: {count:3d} (train: {in_train:3d}, test: {in_test:3d})")


📥 Dataset ORIGINALE caricato:
   Totale sequenze: 371
   Train: 283, Test: 88
   Classi uniche: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

📊 Distribuzione classi (originale):
   [ 0] SQUAT - Correct               :  41 (train:  31, test:  10)
   [ 1] SQUAT - Feet too wide         :  23 (train:  18, test:   5)
   [ 2] SQUAT - Knees inward          :  23 (train:  18, test:   5)
   [ 3] SQUAT - Not low enough        :  21 (train:  17, test:   4)
   [ 4] SQUAT - Front bended          :  24 (train:  17, test:   7)
   [ 5] SQUAT - Unknown               :   9 (train:   9, test:   0)
   [ 6] LUNGES - Correct              :  46 (train:  34, test:  12)
   [ 7] LUNGES - Not low enough       :  40 (train:  30, test:  10)
   [ 8] LUNGES - Knees pass toes      :  41 (train:  31, test:  10)
   [ 9] PLANK - Correct               :  33 (train:  26, test:   7)
   [10] PLANK - Banana back           :  30 (train:  21, test:   9)
   [11] PLANK - Rolled back           :  40 (train:  31, test:   9)


## 3. Identify and Filter Unknown Sequences


In [4]:
# Identifica sequenze Unknown (label == 5)
unknown_mask = labels_orig == 5
unknown_indices = np.where(unknown_mask)[0]

print(f"🔍 Sequenze UNKNOWN (label=5, SQUAT-Unknown):")
print(f"   Totale: {len(unknown_indices)} sequenze")
print(f"\n   Dettaglio:")
for idx in unknown_indices:
    m = meta_orig[idx]
    print(f"   [{idx:3d}] {m['exercise']} - {m['instruction_name']} - Subject: {m['subject']} - Trial: {m['trial_id']}")

# Verifica: tutte le unknown dovrebbero essere SQUAT con instruction_id=10
all_squat = all(meta_orig[idx]['exercise'] == 'SQUAT' for idx in unknown_indices)
all_instr_10 = all(meta_orig[idx]['instruction_id'] == 10 for idx in unknown_indices)
print(f"\n✅ Verifica: tutte SQUAT? {all_squat}")
print(f"✅ Verifica: tutte instruction_id=10? {all_instr_10}")


🔍 Sequenze UNKNOWN (label=5, SQUAT-Unknown):
   Totale: 9 sequenze

   Dettaglio:
   [331] SQUAT - Unknown - Subject: Sena - Trial: 1
   [332] SQUAT - Unknown - Subject: Sena - Trial: 2
   [333] SQUAT - Unknown - Subject: Sena - Trial: 3
   [334] SQUAT - Unknown - Subject: Sena - Trial: 4
   [335] SQUAT - Unknown - Subject: Sena - Trial: 5
   [336] SQUAT - Unknown - Subject: Sena - Trial: 6
   [337] SQUAT - Unknown - Subject: Sena - Trial: 7
   [338] SQUAT - Unknown - Subject: Sena - Trial: 8
   [339] SQUAT - Unknown - Subject: Sena - Trial: 9

✅ Verifica: tutte SQUAT? True
✅ Verifica: tutte instruction_id=10? True


In [5]:
# Crea maschera per sequenze VALIDE (no unknown)
valid_mask = labels_orig != 5
valid_indices_orig = np.where(valid_mask)[0]

print(f"\n📊 Filtraggio:")
print(f"   Originale: {len(sequences_orig)} sequenze")
print(f"   Rimosse (Unknown): {len(unknown_indices)} sequenze")
print(f"   Valide: {len(valid_indices_orig)} sequenze")

# Verifica conteggio atteso (paper: 362)
assert len(valid_indices_orig) == 362, f"Atteso 362, trovato {len(valid_indices_orig)}"
print(f"\n✅ Conteggio corretto: 362 sequenze (come da paper)")



📊 Filtraggio:
   Originale: 371 sequenze
   Rimosse (Unknown): 9 sequenze
   Valide: 362 sequenze

✅ Conteggio corretto: 362 sequenze (come da paper)


## 4. Create Filtered Dataset with Remapped Labels


In [6]:
# Crea nuovo dataset filtrato
sequences_new = []
labels_new = []
labels_old = []  # Mantieni anche le label originali per riferimento
meta_new = []

# Mapping old_index -> new_index
old_to_new_index = {}

for new_idx, old_idx in enumerate(valid_indices_orig):
    old_to_new_index[old_idx] = new_idx
    
    # Sequenza
    sequences_new.append(sequences_orig[old_idx])
    
    # Label: rimappa
    old_label = labels_orig[old_idx]
    new_label = OLD_TO_NEW_LABEL_MAP[old_label]
    labels_new.append(new_label)
    labels_old.append(old_label)
    
    # Meta: copia e aggiorna
    meta_item = meta_orig[old_idx].copy()
    meta_item['old_global_label_id'] = old_label
    meta_item['global_label_id'] = new_label
    meta_item['global_label_name'] = ID_TO_NAME_NEW[new_label]
    meta_new.append(meta_item)

labels_new = np.array(labels_new, dtype=np.int64)
labels_old = np.array(labels_old, dtype=np.int64)

print(f"📊 Dataset NUOVO (no unknown):")
print(f"   Sequenze: {len(sequences_new)}")
print(f"   Labels shape: {labels_new.shape}")
print(f"   Classi uniche: {sorted(np.unique(labels_new).tolist())}")

# Verifica distribuzione
print(f"\n📊 Distribuzione classi (nuovo):")
for cid in range(11):
    count = np.sum(labels_new == cid)
    print(f"   [{cid:2d}] {ID_TO_NAME_NEW[cid]:30s}: {count:3d}")


📊 Dataset NUOVO (no unknown):
   Sequenze: 362
   Labels shape: (362,)
   Classi uniche: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

📊 Distribuzione classi (nuovo):
   [ 0] SQUAT - Correct               :  41
   [ 1] SQUAT - Feet too wide         :  23
   [ 2] SQUAT - Knees inward          :  23
   [ 3] SQUAT - Not low enough        :  21
   [ 4] SQUAT - Front bended          :  24
   [ 5] LUNGES - Correct              :  46
   [ 6] LUNGES - Not low enough       :  40
   [ 7] LUNGES - Knees pass toes      :  41
   [ 8] PLANK - Correct               :  33
   [ 9] PLANK - Banana back           :  30
   [10] PLANK - Rolled back           :  40


## 5. Update Cross-Subject Split


In [7]:
# Aggiorna gli indici dello split (rimuovi unknown, rimappa)
# Gli indici originali che erano unknown non saranno nel nuovo dataset

train_indices_new = []
test_indices_new = []

for old_idx in train_indices_orig:
    if old_idx in old_to_new_index:  # Non è unknown
        train_indices_new.append(old_to_new_index[old_idx])

for old_idx in test_indices_orig:
    if old_idx in old_to_new_index:  # Non è unknown
        test_indices_new.append(old_to_new_index[old_idx])

train_indices_new = np.array(train_indices_new)
test_indices_new = np.array(test_indices_new)

print(f"📊 Split NUOVO:")
print(f"   Train: {len(train_indices_new)} (originale: {len(train_indices_orig)})")
print(f"   Test: {len(test_indices_new)} (originale: {len(test_indices_orig)})")
print(f"   Rimossi dal train: {len(train_indices_orig) - len(train_indices_new)}")
print(f"   Rimossi dal test: {len(test_indices_orig) - len(test_indices_new)}")

# Verifica: nel test non c'erano unknown (Subject 4 = Vidit non ha Unknown)
# Le 9 unknown erano tutte di Sena (Subject 3) nel train
print(f"\n✅ Verifica: Unknown erano tutti nel train (9 sequenze di Sena)")

# Distribuzione per classe nel nuovo split
train_labels_new = labels_new[train_indices_new]
test_labels_new = labels_new[test_indices_new]

print(f"\n📊 Distribuzione classi nel nuovo split:")
print(f"{'Classe':<35} {'Train':>8} {'Test':>8} {'Tot':>8}")
print("-" * 60)
for cid in range(11):
    train_c = np.sum(train_labels_new == cid)
    test_c = np.sum(test_labels_new == cid)
    print(f"{ID_TO_NAME_NEW[cid]:<35} {train_c:>8} {test_c:>8} {train_c+test_c:>8}")


📊 Split NUOVO:
   Train: 274 (originale: 283)
   Test: 88 (originale: 88)
   Rimossi dal train: 9
   Rimossi dal test: 0

✅ Verifica: Unknown erano tutti nel train (9 sequenze di Sena)

📊 Distribuzione classi nel nuovo split:
Classe                                 Train     Test      Tot
------------------------------------------------------------
SQUAT - Correct                           31       10       41
SQUAT - Feet too wide                     18        5       23
SQUAT - Knees inward                      18        5       23
SQUAT - Not low enough                    17        4       21
SQUAT - Front bended                      17        7       24
LUNGES - Correct                          34       12       46
LUNGES - Not low enough                   30       10       40
LUNGES - Knees pass toes                  31       10       41
PLANK - Correct                           26        7       33
PLANK - Banana back                       21        9       30
PLANK - Rolled back 

## 6. Verify Alignment with Paper


In [8]:
# Verifica allineamento con paper
# Paper totals: Squats=132, Lunges=127, Planks=103, Total=362

# Conta per exercise
exercise_counts = {'Squats': 0, 'Lunges': 0, 'Planks': 0}

for m in meta_new:
    ex = m['exercise']
    if ex == 'SQUAT':
        exercise_counts['Squats'] += 1
    elif ex == 'Lunges':
        exercise_counts['Lunges'] += 1
    elif ex == 'Plank':
        exercise_counts['Planks'] += 1

print("📋 VERIFICA ALLINEAMENTO CON PAPER")
print("=" * 50)
print(f"{'Exercise':<15} {'Paper':>10} {'Data':>10} {'Match':>10}")
print("-" * 50)

paper_totals = {'Squats': 132, 'Lunges': 127, 'Planks': 103}
all_match = True
for ex, paper_count in paper_totals.items():
    data_count = exercise_counts[ex]
    match = "✅" if paper_count == data_count else "❌"
    if paper_count != data_count:
        all_match = False
    print(f"{ex:<15} {paper_count:>10} {data_count:>10} {match:>10}")

total_paper = sum(paper_totals.values())
total_data = sum(exercise_counts.values())
match = "✅" if total_paper == total_data else "❌"
print("-" * 50)
print(f"{'TOTALE':<15} {total_paper:>10} {total_data:>10} {match:>10}")

if all_match:
    print("\n✅ TUTTI I CONTEGGI CORRISPONDONO AL PAPER!")
else:
    print("\n❌ MISMATCH RILEVATO - verificare i dati")


📋 VERIFICA ALLINEAMENTO CON PAPER
Exercise             Paper       Data      Match
--------------------------------------------------
Squats                 132        132          ✅
Lunges                 127        127          ✅
Planks                 103        103          ✅
--------------------------------------------------
TOTALE                 362        362          ✅

✅ TUTTI I CONTEGGI CORRISPONDONO AL PAPER!


## 7. Save New Dataset


In [9]:
# === SAVE NEW DATASET ===

# 1. Save sequences pickle
ec3d_no_unknown = {
    'sequences': sequences_new,
    'labels': labels_new,
    'labels_old': labels_old,  # Per riferimento
    'meta': meta_new,
    'old_to_new_label_map': OLD_TO_NEW_LABEL_MAP,
    'id_to_name': ID_TO_NAME_NEW,
    'num_classes': 11,
}

pkl_path = DATA_DIR / "ec3d_sequences_no_unknown.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(ec3d_no_unknown, f)
print(f"✅ Salvato: {pkl_path}")

# 2. Save split JSON
split_no_unknown = {
    'train_subjects': split_orig['train_subjects'],
    'test_subjects': split_orig['test_subjects'],
    'train_indices': train_indices_new.tolist(),
    'test_indices': test_indices_new.tolist(),
}

split_path = DATA_DIR / "split_cross_subject_no_unknown.json"
with open(split_path, "w") as f:
    json.dump(split_no_unknown, f, indent=2)
print(f"✅ Salvato: {split_path}")

# 3. Save info JSON
info = {
    'description': 'EC3D dataset without Unknown class (paper-aligned)',
    'created': datetime.now().isoformat(),
    'num_sequences': len(sequences_new),
    'num_classes': 11,
    'train_sequences': len(train_indices_new),
    'test_sequences': len(test_indices_new),
    'removed_class': {
        'old_id': 5,
        'name': 'SQUAT - Unknown',
        'count_removed': len(unknown_indices),
    },
    'label_mapping': {str(k): v for k, v in OLD_TO_NEW_LABEL_MAP.items()},
    'class_names': ID_TO_NAME_NEW,
    'counts_per_exercise': exercise_counts,
    'paper_expected': paper_totals,
}

info_path = DATA_DIR / "ec3d_no_unknown_info.json"
with open(info_path, "w") as f:
    json.dump(info, f, indent=2)
print(f"✅ Salvato: {info_path}")


✅ Salvato: /home/giov/Scrivania/Tesi/pose-text-feedback-thesis/data/EC3D/ec3d_sequences_no_unknown.pkl
✅ Salvato: /home/giov/Scrivania/Tesi/pose-text-feedback-thesis/data/EC3D/split_cross_subject_no_unknown.json
✅ Salvato: /home/giov/Scrivania/Tesi/pose-text-feedback-thesis/data/EC3D/ec3d_no_unknown_info.json


## 8. Summary


In [10]:
# === SUMMARY ===
print("=" * 70)
print("📋 RIEPILOGO FINALE")
print("=" * 70)

print(f"""
📊 DATASET EC3D NO-UNKNOWN:
   - Sequenze totali: {len(sequences_new)} (originale: {len(sequences_orig)})
   - Classi: 11 (originale: 12)
   - Rimossi: {len(unknown_indices)} sequenze di classe "Unknown"
   
📂 FILE CREATI:
   - {pkl_path.name}
   - {split_path.name}
   - {info_path.name}

🎯 ALLINEAMENTO CON PAPER:
   - Squats: {exercise_counts['Squats']}/132 ✅
   - Lunges: {exercise_counts['Lunges']}/127 ✅
   - Planks: {exercise_counts['Planks']}/103 ✅
   - TOTALE: {total_data}/362 ✅

📋 SPLIT CROSS-SUBJECT:
   - Train: {len(train_indices_new)} sequenze
   - Test: {len(test_indices_new)} sequenze (invariato - Unknown era solo nel train)

🔄 LABEL MAPPING (old -> new):
   - 0-4 → 0-4 (Squat)
   - 5 → RIMOSSO (Unknown)
   - 6-11 → 5-10 (Lunges + Planks)

💡 USO:
   from utils import load_ec3d
   data = load_ec3d('data/EC3D', no_unknown=True)
""")

print("=" * 70)
print("✅ DATASET NO-UNKNOWN CREATO CON SUCCESSO!")
print("=" * 70)


📋 RIEPILOGO FINALE

📊 DATASET EC3D NO-UNKNOWN:
   - Sequenze totali: 362 (originale: 371)
   - Classi: 11 (originale: 12)
   - Rimossi: 9 sequenze di classe "Unknown"

📂 FILE CREATI:
   - ec3d_sequences_no_unknown.pkl
   - split_cross_subject_no_unknown.json
   - ec3d_no_unknown_info.json

🎯 ALLINEAMENTO CON PAPER:
   - Squats: 132/132 ✅
   - Lunges: 127/127 ✅
   - Planks: 103/103 ✅
   - TOTALE: 362/362 ✅

📋 SPLIT CROSS-SUBJECT:
   - Train: 274 sequenze
   - Test: 88 sequenze (invariato - Unknown era solo nel train)

🔄 LABEL MAPPING (old -> new):
   - 0-4 → 0-4 (Squat)
   - 5 → RIMOSSO (Unknown)
   - 6-11 → 5-10 (Lunges + Planks)

💡 USO:
   from utils import load_ec3d
   data = load_ec3d('data/EC3D', no_unknown=True)

✅ DATASET NO-UNKNOWN CREATO CON SUCCESSO!
